
# Mapping `NIDN` & `NIP` dari `data_dosen.csv` ke `dosen.csv` (berdasarkan Nama)
Notebook ini menyatukan kolom **NIDN** (melengkapi jika kosong) dan menambahkan kolom **NIP** ke `dosen.csv` dengan cara **mapping berdasarkan nama**.  
Strategi: **normalisasi nama** (hapus gelar/prefix/suffix, lowercase) agar cocok dengan `data_dosen.csv`.


## 1) Path file

In [28]:
from pathlib import Path
BASE = Path().cwd() # sesuaikan jika perlu
DOSEN_CSV = BASE / "file_tabulars/dosen.csv"
DATA_DOSEN_CSV = BASE / "file_tabulars/data_dosen.csv"
SCIVAL = BASE / "file_tabulars/scival_scraped.csv"

# Output
OUT_MAIN = BASE / "file_tabulars/dosen_data.csv"
OUT_UNMATCHED = BASE / "file_tabulars/unmatched_names.csv"
OUT_CONFLICTS = BASE / "file_tabulars/nidn_conflicts.csv"
OUT_AMBIG = BASE / "file_tabulars/ambiguous_name_groups.csv"

print(DOSEN_CSV, DATA_DOSEN_CSV,SCIVAL, sep="\n")


C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\dosen.csv
C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\data_dosen.csv
C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\scival_scraped.csv


## 2) Import & Helper function

In [29]:
import pandas as pd
import re
import unicodedata
from collections import Counter

def pick_col(df, candidates, required=False):
    """Ambil kolom pertama yang tersedia dari daftar kandidat."""
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise ValueError(f"Couldn't find any of these columns: {candidates}")
    return None

def strip_accents(s: str) -> str:
    """Hapus aksen/diakritik (NFD)."""
    return "".join(ch for ch in unicodedata.normalize("NFD", s) if unicodedata.category(ch) != "Mn")

# Prefix gelar umum di Indonesia (bisa ditambah sesuai kebutuhan)
TITLE_PREFIX = re.compile(r"^\s*(prof\.?|drs\.?|dra\.?|dr\.?|ir\.?|h\.?|hj\.?)\s+", flags=re.IGNORECASE)

def normalize_name(raw: str) -> str:
    """Normalisasi nama:
    - hapus aksen
    - potong di koma pertama (hapus gelar/sufiks seperti S.Kom., M.Kom.)
    - hapus prefix gelar (Prof., Dr., Ir., H., Hj., Drs., Dra.)
    - hilangkan titik, rapikan spasi, lowercase
    """
    if pd.isna(raw):
        return ""
    s = str(raw).strip()
    s = strip_accents(s)
    # buang bagian setelah koma (umumnya gelar)
    s = s.split(",")[0]
    # hapus prefix gelar berulang sampai bersih
    prev = None
    while prev != s:
        prev = s
        s = TITLE_PREFIX.sub("", s).strip()
    # hilangkan titik pada inisial
    s = re.sub(r"\b([A-Z])\.\s*", r"\1 ", s)
    # ganti sisa titik jadi spasi, rapikan spasi
    s = s.replace(".", " ")
    s = re.sub(r"\s+", " ", s)
    return s.lower().strip()

def first_mode_or_first(series: pd.Series):
    """Pilih nilai paling sering (mode). Jika seri kosong, NA; jika seri ada tie, ambil yang muncul duluan."""
    s = series.dropna().astype(str)
    if s.empty:
        return pd.NA
    counts = Counter(s)
    # urut: frekuensi desc, lalu posisi kemunculan pertama asc
    best = sorted(counts.items(), key=lambda kv: (-kv[1], s.tolist().index(kv[0])))[0][0]
    return best


## 3) Load data

In [31]:

dosen = pd.read_csv(DOSEN_CSV, encoding="utf-8-sig")
data_dosen = pd.read_csv(DATA_DOSEN_CSV, encoding="utf-8-sig")
scival         = pd.read_csv(SCIVAL, encoding="utf-8-sig")

# --- Filter out unwanted rows ---
# Drop dosen yang memiliki program studi 'Pendidikan Teknik Elektro' (case-insensitive)
if "nama_prodi" in dosen.columns:
    before = len(dosen)
    dosen = dosen[~dosen["nama_prodi"]
                  .astype(str)
                  .str.lower()
                  .str.contains("pendidikan teknik elektro", na=False)]
    print(f"Dropped {before - len(dosen)} rows from Pendidikan Teknik Elektro")
else:
    print("Kolom 'nama_prodi' tidak ditemukan di dosen.csv — lewati langkah drop.")

print("dosen.csv cols:", list(dosen.columns))
print("data_dosen.csv cols:", list(data_dosen.columns))
print("\nShapes:", dosen.shape, data_dosen.shape, scival.shape)


Dropped 14 rows from Pendidikan Teknik Elektro
dosen.csv cols: ['id_sdm', 'nama_dosen', 'nidn', 'jabatan_akademik', 'pendidikan_tertinggi', 'status_ikatan_kerja', 'status_aktivitas', 'nama_pt', 'nama_prodi', 'jumlah_penelitian', 'jumlah_pengabdian', 'jumlah_karya_ilmiah', 'jumlah_paten']
data_dosen.csv cols: ['Nama', 'NIDN', 'NIP', 'Detail_URL', 'Photo_URL', 'QR_Code_URL', 'ID_Sinta', 'ID_Scopus', 'ID_Scholar', 'Program_Studi']

Shapes: (129, 13) (1733, 10) (2536, 69)


## 4) Identifikasi kolom penting

In [32]:

# Nama dosen di kedua file (wajib ditemukan salah satu kandidat)
dosen_name_col = pick_col(dosen, ["nama", "nama_dosen", "Nama", "Nama Dosen"], required=True)
data_name_col  = pick_col(data_dosen, ["nama_dosen", "nama", "Nama Dosen", "Nama"], required=True)

# Identitas dari data_dosen (wajib)
nidn_col = pick_col(data_dosen, ["nidn", "NIDN"], required=True)
nip_col  = pick_col(data_dosen, ["nip", "NIP"], required=True)

# Apakah dosen.csv sudah punya nidn?
has_nidn_in_dosen = "nidn" in dosen.columns
dosen_nidn_col = "nidn" if has_nidn_in_dosen else None

dosen_name_col, data_name_col, nidn_col, nip_col, dosen_nidn_col


('nama_dosen', 'Nama', 'NIDN', 'NIP', 'nidn')

## 5) Normalisasi nama → `_norm_name`

In [20]:

dosen = dosen.copy()
data_dosen = data_dosen.copy()

dosen["_norm_name"] = dosen[dosen_name_col].map(normalize_name)
data_dosen["_norm_name"] = data_dosen[data_name_col].map(normalize_name)

dosen[[dosen_name_col, "_norm_name"]].head(10)


,nama_dosen,_norm_name
0,Nurul Laili,nurul laili
1,Sabrina Amelialevi,sabrina amelialevi
2,Atik Wintarti,atik wintarti
3,Siska Puspitaningsih,siska puspitaningsih
4,Heri Purnawan,heri purnawan
5,Belgis Ainatul Iza,belgis ainatul iza
6,Ibnu Febry Kurniawan,ibnu febry kurniawan
7,Dinda Galuh Guminta,dinda galuh guminta
8,Ulfa Siti Nuraini,ulfa siti nuraini
9,Yuliani Puji Astuti,yuliani puji astuti


## 6) Tabel mapping dari `data_dosen.csv` (mode nilai per nama-normal)

In [21]:

# Jika ada kolom 'id' bagus untuk menghitung group size; jika tidak, pakai kolom nama
count_key = "id" if "id" in data_dosen.columns else data_name_col

map_df = (
    data_dosen.groupby("_norm_name", dropna=False)
    .agg(
        nidn_mapped=(nidn_col, first_mode_or_first),
        nip_mapped=(nip_col, first_mode_or_first),
        sample_full_name=(data_name_col, lambda s: next((x for x in s if pd.notna(x)), pd.NA)),
        count=(count_key, "count"),
    )
    .reset_index()
)
map_df.head(10)


,_norm_name,nidn_mapped,nip_mapped,sample_full_name,count
0,(h c ) h abdul halim iskandar,8996950022.0,202203004,"Dr. (H.C.) H. Abdul Halim Iskandar, M.Pd.",1
1,a burhanuddin kusuma nugraha,31129107.0,199112312023211044,"A Burhanuddin Kusuma Nugraha, S.Pd., M.Kes.",1
2,a grummy wailanduw,23086203.0,196208231986011001,"Dr. A. Grummy Wailanduw, M.Pd., M.T.",1
3,a'rasy fahrullah,4108109.0,198110042015041001,"Dr. A'rasy Fahrullah, S.Sos., M.Si.",1
4,a'yunin sofro,23088002.0,198008232005012002,"A'yunin Sofro, M.Si., Ph.D.",1
5,abadi,30086501.0,196508301991011001,"Prof. Dr. Abadi, M.Sc.",4
6,abd kholiq,23057702.0,197705232000121001,"Abd. Kholiq, S.Pd., M.T.",1
7,abdiyah amudi,730078601.0,198607302019122001,"Abdiyah Amudi, S.T., M.T.",1
8,abdul aziz hakim,1038301.0,198303012005011001,"Dr. Abdul Aziz Hakim, S.Or., M.Or.",1
9,abdul aziz khoiri,21129308.0,199312212023211014,"Abdul Aziz Khoiri, S.Pd., M.Pd.",1


## 7) Join ke `dosen.csv` dan isi kolom `nidn`/`nip`

In [22]:

merged = dosen.merge(map_df, how="left", on="_norm_name")

# Siapkan kolom NIDN akhir
if not has_nidn_in_dosen:
    merged["nidn"] = merged["nidn_mapped"]
else:
    # pertahankan yang sudah ada; yang kosong diisi dari mapping
    merged["nidn"] = merged["nidn"].astype(object)
    need_fill = merged["nidn"].isna() | (merged["nidn"].astype(str).str.strip() == "")
    merged.loc[need_fill, "nidn"] = merged.loc[need_fill, "nidn_mapped"]

# Kolom NIP baru (langsung dari mapping)
merged["nip"] = merged["nip_mapped"]

# Susun kolom agar kolom baru di belakang
ordered_cols = list(dosen.columns)
for c in ["nidn", "nip", "sample_full_name"]:
    if c not in ordered_cols and c in merged.columns:
        ordered_cols.append(c)

final = merged[ordered_cols].copy()
final.head(10)


,id_sdm,nama_dosen,nidn,jabatan_akademik,pendidikan_tertinggi,status_ikatan_kerja,status_aktivitas,nama_pt,nama_prodi,jumlah_penelitian,jumlah_pengabdian,jumlah_karya_ilmiah,jumlah_paten,_norm_name,nip,sample_full_name
0,sLZzYTPWD4CeqSEAVN-2cT_TIGHN2WrXWbgM3KWj3vUrDC...,Nurul Laili,<NA>,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,0,0,0,0,nurul laili,202509238,"Nurul Laili, S.Pd., M.Sc."
1,RmxdOmO31i_M8sDBvKw_GMni0V6WrrsplULBC16QoFHhsp...,Sabrina Amelialevi,<NA>,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,0,0,0,0,sabrina amelialevi,202509217,"Sabrina Amelialevi, S.Kom., M.Kom."
2,e4u0fxTGIXZk6OVg1D7m069hCuRXliun1mKiHaKuuctsYH...,Atik Wintarti,12106608.0,Lektor Kepala,S3,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,11,6,74,12,atik wintarti,196610121991032002,"Dr. Atik Wintarti, M.Kom."
3,ug18IBRkRo2_1vVjCnaL2pNGZVIi9vtromIBpKjtE9X1x2...,Siska Puspitaningsih,725129501.0,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,0,4,11,1,siska puspitaningsih,202509216,"Siska Puspitaningsih, S.Kom., M.Kom."
4,XSZZdZfK-0lj0Q0D8ItXeI3YO8eS_s4kerNAxQfMPBwYdG...,Heri Purnawan,706069301.0,NaN,S3,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,0,4,12,1,heri purnawan,202509200,"Dr. Heri Purnawan, S.Si., M.Si."
5,FYVyQepNiSlB6DaekFokRICIT-HzvzlvimBRQxVrrnEmFJ...,Belgis Ainatul Iza,<NA>,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,0,0,0,0,belgis ainatul iza,202509237,"Belgis Ainatul Iza, S.Si, M.Mat."
6,51rEX5Z49Gjhoqhh3pOnCQ5S2QvV3JYEACF0lOljO1g5qu...,Ibnu Febry Kurniawan,18028801.0,Asisten Ahli,S3,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,12,12,26,2,ibnu febry kurniawan,198802182015041001,"Ibnu Febry Kurniawan, S.Kom., M.Sc., Ph.D."
7,UvM-UyzM2-1YAa5s4zKACptXeqMUvR5JVzt4Ioo2VFG3pi...,Dinda Galuh Guminta,11129602.0,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,2,0,0,1,dinda galuh guminta,199612112024062003,"Dinda Galuh Guminta, M.Stat."
8,PGt25k6rqQshJfmGoStYk8yF1W2TgWV3Gfg8tRkOblCdSm...,Ulfa Siti Nuraini,4109601.0,NaN,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,4,3,4,0,ulfa siti nuraini,202310142,"Ulfa Siti Nuraini, S.Stat., M.Stat."
9,5HLmrZPYpQ7E_bYFJPAMzgtI2hNLPhER5JBmHJ8dB53rVo...,Yuliani Puji Astuti,31077804.0,Lektor,S2,Dosen Tetap,Aktif,UNIVERSITAS NEGERI SURABAYA,SAINS DATA,12,9,32,9,yuliani puji astuti,197807312006042001,"Yuliani Puji Astuti, S.Si., M.Si."


## 8) Diagnostik cepat (unmatched, konflik, ambigu)

In [23]:

# Unmatched: tidak dapat nidn & nip dari mapping
unmatched = merged[merged["nidn_mapped"].isna() & merged["nip_mapped"].isna()][[dosen_name_col, "_norm_name"]].copy()

# Konflik NIDN: jika dosen.csv sudah punya NIDN dan berbeda dengan hasil mapping
if has_nidn_in_dosen:
    conflicts = merged[
        merged["nidn"].notna() & merged["nidn_mapped"].notna() &
        (merged["nidn"].astype(str).str.strip() != merged["nidn_mapped"].astype(str).str.strip())
    ][[dosen_name_col, "nidn", "nidn_mapped", "_norm_name"]].copy()
else:
    conflicts = pd.DataFrame(columns=[dosen_name_col, "nidn", "nidn_mapped", "_norm_name"])

# Ambiguous: group size > 1 di data_dosen (nama-normal sama muncul >1)
ambiguous = map_df[map_df["count"] > 1][["_norm_name", "sample_full_name", "nidn_mapped", "nip_mapped", "count"]].copy()

summary = {
    "total_dosen_rows": len(dosen),
    "mapped_by_name": int(merged["nidn_mapped"].notna().sum()),
    "added_nip": int(merged["nip"].notna().sum()),
    "filled_missing_nidn_from_mapping": int(((merged["nidn_mapped"].notna()) & (dosen["nidn"] if has_nidn_in_dosen else pd.Series([pd.NA]*len(dosen))).isna()).sum()) if has_nidn_in_dosen else int(merged["nidn"].notna().sum()),
    "unmatched_names": len(unmatched),
    "nidn_conflicts": len(conflicts),
    "ambiguous_groups": len(ambiguous),
}

summary, unmatched.head(10), conflicts.head(10), ambiguous.head(10)


({'total_dosen_rows': 129,
  'mapped_by_name': 66,
  'added_nip': 72,
  'filled_missing_nidn_from_mapping': 3,
  'unmatched_names': 57,
  'nidn_conflicts': 0,
  'ambiguous_groups': 15},
              nama_dosen           _norm_name
 28    Bambang Suprianto    bambang suprianto
 30          Aidil Danas          aidil danas
 31              Budiman              budiman
 32         Sri Indriani         sri indriani
 33  Rusvaira Qatrunnada  rusvaira qatrunnada
 40         Hazlif Nazif         hazlif nazif
 41         Rosnita Rauf         rosnita rauf
 42    Supriyana Nugroho    supriyana nugroho
 43           Ari Wibowo           ari wibowo
 44        Slamet Riyadi        slamet riyadi,
 Empty DataFrame
 Columns: [nama_dosen, nidn, nidn_mapped, _norm_name]
 Index: [],
                 _norm_name                               sample_full_name  \
 5                    abadi                         Prof. Dr. Abadi, M.Sc.   
 26            achmad lutfi                  Prof. Dr. Achmad Lutfi,

## 9) Simpan hasil dan laporan

In [25]:

final.to_csv(OUT_MAIN, index=False, encoding="utf-8-sig")
unmatched.to_csv(OUT_UNMATCHED, index=False, encoding="utf-8-sig")
conflicts.to_csv(OUT_CONFLICTS, index=False, encoding="utf-8-sig")
ambiguous.to_csv(OUT_AMBIG, index=False, encoding="utf-8-sig")

print("Saved:")
print("-", OUT_MAIN)
print("-", OUT_UNMATCHED)
print("-", OUT_CONFLICTS)
print("-", OUT_AMBIG)


Saved:
- C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\dosen_data.csv
- C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\unmatched_names.csv
- C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\nidn_conflicts.csv
- C:\Users\rizky\OneDrive\Dokumen\GitHub\Tugas_Akhir\scraping\file_tabulars\ambiguous_name_groups.csv



## 10) Opsi penyempurnaan (opsional)
- Tambah daftar gelar yang dihapus (mis. **S.Kom, M.Kom, M.T., M.Sc., Ph.D.** walau tanpa koma).
- Kamus ejaan (contoh: *muhammad ↔ muhamad*, *muhamad ↔ mochammad*).
- Disambiguasi nama dengan atribut tambahan (mis. `nama_pt`, `program_studi`, tahun lahir) jika tersedia.
